<h2 align=left style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazir" color="#0099cc">
0. Import Libraries

</font>
</h2>

In [1]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import MinMaxScaler
import keras
from keras.layers import Input, Embedding, Flatten, Dot, Activation
from keras.models import Model

<h2 align=left style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazir" color="#0099cc">
1. Data Loading
</font>
</h2>

In [3]:
# Loading the dataset
valid = pd.read_csv('amazon_valid.csv')
train = pd.read_csv('amazon_train.csv')
test = pd.read_csv('amazon_test.csv')
train.head(5)

,UserID,ProductID,Rating,Timestamp
0,A3HICVLF4PFFMN,0594481813,5.0,2014-05-05
1,A2QBZA4S1ROX9Q,0594481813,3.0,2013-05-25
2,AT09WGFUM934H,0594481813,3.0,2013-08-31
3,AGAKHE014LQFU,0594481813,3.0,2013-09-18
4,A1S6B5QFWGVL5U,0594481813,4.0,2013-06-27


<h2 align=left style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazir" color="#0099cc">
2. Preprocessing
</font>
</h2>



In [4]:
product_le = LabelEncoder()
user_le = LabelEncoder()

train['ProductEnc'] = product_le.fit_transform(train['ProductID'])
train['UserEnc'] = user_le.fit_transform(train['UserID'])

# if test exists
test['ProductEnc'] = product_le.transform(test['ProductID'])
test['UserEnc'] = user_le.transform(test['UserID'])
valid['UserEnc'] = user_le.transform(valid['UserID'])
valid['ProductEnc'] = product_le.transform(valid['ProductID'])


In [5]:
train.head(20)

,UserID,ProductID,Rating,Timestamp,ProductEnc,UserEnc
0,A3HICVLF4PFFMN,0594481813,5.0,2014-05-05,0,59277
1,A2QBZA4S1ROX9Q,0594481813,3.0,2013-05-25,0,41167
2,AT09WGFUM934H,0594481813,3.0,2013-08-31,0,85549
3,AGAKHE014LQFU,0594481813,3.0,2013-09-18,0,77047
4,A1S6B5QFWGVL5U,0594481813,4.0,2013-06-27,0,18672
5,A2IDCSC6NVONIZ,0972683275,5.0,2013-04-30,1,35949
6,A3BMUBUC1N77U8,0972683275,4.0,2013-11-23,1,55248
7,AVRFGGCCCR6QU,0972683275,4.0,2010-08-30,1,87318
8,AYQNWE3AX4H08,0972683275,5.0,2013-07-18,1,89229
9,A3P24ZSF9IQRJJ,0972683275,5.0,2014-06-13,1,64167


In [6]:
scaler = MinMaxScaler()

train['Rating'] = scaler.fit_transform(train[['Rating']])
valid['Rating'] = scaler.fit_transform(valid[['Rating']])

<h2  align=left style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazir" color="#0099cc">
3. Model Development
</font>
</h2>

In [7]:
n_users = train['UserEnc'].max() + 1
n_products = train['ProductEnc'].max() + 1
embedding_size = 50

user_input = Input(shape=(1,), name='user_input')
product_input = Input(shape=(1,), name='product_input')

user_embedding = Embedding(n_users, embedding_size)(user_input)
product_embedding = Embedding(n_products, embedding_size)(product_input)

user_vec = Flatten()(user_embedding)
product_vec = Flatten()(product_embedding)

dot_user_product = Dot(axes=1)([user_vec, product_vec])
output = Activation('sigmoid')(dot_user_product)

model = Model(inputs=[user_input, product_input], outputs=output)

model.compile(
    optimizer='adam',
    loss='binary_crossentropy'
)
model.summary()


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ user_input          │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ product_input       │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, 1, 50)     │  4,502,500 │ user_input[0][0]  │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_1         │ (None, 1, 50)     │  2,438,450 │ product_input[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten (Flatten)   │ (None, 50)        │          0 │ embedding[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_1 (Flatten) │ (None, 50)        │          0 │ embedding_1[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dot (Dot)           │ (None, 1)         │          0 │ flatten[0][0],    │
│                     │                   │            │ flatten_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation          │ (None, 1)         │          0 │ dot[0][0]         │
│ (Activation)        │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 6,940,950 (26.48 MB)

 Trainable params: 6,940,950 (26.48 MB)

 Non-trainable params: 0 (0.00 B)

<h2  align=left style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazir" color="#0099cc">
4. Training
</font>
</h2>

In [8]:
history = model.fit(
    [train['UserEnc'], train['ProductEnc']],
    train['Rating'],
    epochs=10,
    batch_size=32,
    validation_split=0.1
)


Epoch 1/10
25572/25572 ━━━━━━━━━━━━━━━━━━━━ 1952s 76ms/step - loss: 0.6899 - val_loss: 0.6934
Epoch 2/10
25572/25572 ━━━━━━━━━━━━━━━━━━━━ 1471s 58ms/step - loss: 0.5141 - val_loss: 0.6938
Epoch 3/10
25572/25572 ━━━━━━━━━━━━━━━━━━━━ 1316s 51ms/step - loss: 0.3638 - val_loss: 0.6941
Epoch 4/10
25572/25572 ━━━━━━━━━━━━━━━━━━━━ 1652s 65ms/step - loss: 0.2943 - val_loss: 0.6943
Epoch 5/10
25572/25572 ━━━━━━━━━━━━━━━━━━━━ 1660s 65ms/step - loss: 0.2574 - val_loss: 0.6946
Epoch 6/10
25572/25572 ━━━━━━━━━━━━━━━━━━━━ 1661s 65ms/step - loss: 0.2383 - val_loss: 0.6948
Epoch 7/10
25572/25572 ━━━━━━━━━━━━━━━━━━━━ 1631s 64ms/step - loss: 0.2290 - val_loss: 0.6950
Epoch 8/10
25572/25572 ━━━━━━━━━━━━━━━━━━━━ 1632s 64ms/step - loss: 0.2245 - val_loss: 0.6952
Epoch 9/10
25572/25572 ━━━━━━━━━━━━━━━━━━━━ 1675s 65ms/step - loss: 0.2211 - val_loss: 0.6954
Epoch 10/10
25572/25572 ━━━━━━━━━━━━━━━━━━━━ 1364s 52ms/step - loss: 0.2196 - val_loss: 0.6955


<h2  align=left style="line-height:200%;font-family:vazir;color:#0099cc">
<font face="vazir" color="#0099cc">
5. Evaluation
</font>
</h2>



In [9]:
from sklearn.metrics import precision_score

y_prob = model.predict([valid['UserEnc'], valid['ProductEnc']])
y_pred = (y_prob >= 0.5).astype(int).ravel()
valid['Label'] = (valid['Rating'] >= 0.75).astype(int)
precision = precision_score(valid['Label'], y_pred)
print("Validation Precision:", precision)


2810/2810 ━━━━━━━━━━━━━━━━━━━━ 4s 1ms/step
Validation Precision: 0.8298645523620747
